In [1]:
%load_ext autoreload
%autoreload 2

import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("optuna").setLevel(logging.WARNING)

import numpy as np
import pandas as pd
import optuna
import MetaTrader5 as mt5
from sklearn.ensemble import IsolationForest

from trading_engine import (
    get_mt5_data, split_data, apply_trading_costs,
    apply_sl_tp_and_costs, calculate_ai_metrics, generate_tearsheet,
)
from instruments import get_market_watch_symbols  # optional symbol scanner

# quant_toolbox also exposes: fit_strategy, run_backtest, stress_test, walk_forward,
# get_swap_pct, get_ref_atr, add_atr, STOP_MODES, ...
from quant_toolbox import *

optuna.logging.set_verbosity(optuna.logging.WARNING)

if not mt5.initialize():
    raise RuntimeError(f"MT5 initialize() failed: {mt5.last_error()}")
print("MT5 connected:", mt5.terminal_info().name)


MT5 connected: MetaTrader 5 EXNESS


In [8]:
# ----------------------------- CONFIG -----------------------------
SYMBOLS      = get_market_watch_symbols() # ["US30", "NAS100"]      # adjust to your broker's exact symbol names
TIMEFRAME    = mt5.TIMEFRAME_M5
N_CANDLES    = 50000                  # M1 bars; ~70 trading days if 24h, broker history permitting
TRAIN_PCT    = 0.70
N_TRIALS_PER_VARIANT = 60
MIN_ACTIVE_DAYS = 30
MIN_IS_FITNESS  = 0.0
GRADED_PENALTY  = True
SWAP_DEFAULT    = 0.0002
SEED         = 42

# Article uses a structural stop; trail/TP are unsupported. Restrict to fixed ATR stops.
import quant_toolbox as qt
STOP_MODES = ["fixed"]
qt.FIXED_RANGE = (1.0, 5.0)            # bracket the 15-30 pip structural stop on M1 indices

# >>> STRATEGY SECTION: the ONLY part that changes between strategies <<<
VARIANTS = ["mambafx_ny_breakout"]


def compute_strategy_signals(df, variant, entry_buffer_atr=0.0,
                             entry_cutoff_min=660, exit_min=955):
    """
    Causal approximation of the MambaFX NY-open breakout.

    Pre-market range: 08:30-09:30 America/New_York, computed only from bars
    strictly before 09:30 on the same date.

    Entry: 09:30 <= time < entry_cutoff_min (660 = 11:00 ET), one entry per day.
    Long  if close > PM_High + entry_buffer_atr * ATR_14
    Short if close < PM_Low  - entry_buffer_atr * ATR_14

    Exit: close back inside the pre-market box (fakeout exit) or hard flat at
    exit_min (955 = 15:55 ET). No overnight exposure.

    Returns a held target-position Series: +1 long, -1 short, 0 flat.
    """
    work = df[["open", "high", "low", "close"]].copy()
    n = len(work)

    if n < 100:
        return pd.Series(np.zeros(n, dtype=float), index=df.index, name="position")

    # ---- timezone: df.index is UTC per get_mt5_data; convert to New York ET ----
    idx = df.index
    if idx.tz is None:
        et = idx.tz_localize("UTC").tz_convert("America/New_York")
    else:
        et = idx.tz_convert("America/New_York")

    minutes = et.hour * 60 + et.minute
    dates = et.date

    # ---- pre-market range per date: 08:30 <= time < 09:30 ET ----
    pm_mask = (minutes >= 510) & (minutes < 570)
    if pm_mask.sum() == 0:
        return pd.Series(np.zeros(n, dtype=float), index=df.index, name="position")

    pm_df = pd.DataFrame({
        "date": dates[pm_mask],
        "high": work["high"].to_numpy()[pm_mask],
        "low":  work["low"].to_numpy()[pm_mask],
    })
    pm_range = pm_df.groupby("date").agg(pm_high=("high", "max"),
                                         pm_low=("low", "min"))

    date_series = pd.Series(dates, index=df.index)
    pm_high = date_series.map(pm_range["pm_high"]).to_numpy(dtype=float)
    pm_low = date_series.map(pm_range["pm_low"]).to_numpy(dtype=float)

    # ---- optional “solidly outside” buffer ----
    if entry_buffer_atr and float(entry_buffer_atr) > 0.0:
        atr_df = add_atr(work.copy(), period=14)
        atr = atr_df["ATR_14"].to_numpy(dtype=float)
    else:
        atr = np.zeros(n, dtype=float)

    close = work["close"].to_numpy(dtype=float)

    pos = np.zeros(n, dtype=float)
    s = 0
    entered_today = {}

    for i in range(n):
        d = dates[i]
        m = minutes[i]

        # hard flat after exit_min (15:55 ET) and before the open
        if m >= exit_min:
            s = 0
            pos[i] = s
            continue
        if m < 570:                      # before 09:30 ET
            s = 0
            pos[i] = s
            continue

        # ---- exits: re-entry into the pre-market box = failed breakout ----
        if s == 1 and not np.isnan(pm_high[i]) and close[i] < pm_high[i]:
            s = 0
        elif s == -1 and not np.isnan(pm_low[i]) and close[i] > pm_low[i]:
            s = 0

        # ---- entries: one per day, inside the entry window ----
        if (s == 0 and m < entry_cutoff_min
                and not entered_today.get(d, False)
                and not np.isnan(pm_high[i]) and not np.isnan(pm_low[i])):

            buf = atr[i] * float(entry_buffer_atr) if entry_buffer_atr else 0.0

            if close[i] > pm_high[i] + buf:
                s = 1
                entered_today[d] = True
            elif close[i] < pm_low[i] - buf:
                s = -1
                entered_today[d] = True

        pos[i] = s

    return pd.Series(pos, index=df.index, name="position")


def suggest_params(trial, variants):
    """Strategy params only. Stops are searched separately by fit_strategy."""
    p = {"variant": trial.suggest_categorical("variant", variants)}
    p["entry_buffer_atr"] = trial.suggest_float("entry_buffer_atr", 0.0, 0.5)
    return p


def map_params(p):
    """Flat Optuna params -> keyword arguments of compute_strategy_signals."""
    return {
        "variant": p["variant"],
        "entry_buffer_atr": p["entry_buffer_atr"],
    }
# >>> END STRATEGY SECTION <<<

# ----------------------- SCREEN: optimise IS -> blind OOS -> cost stress -----------------------
portfolio_results, stress_tables = [], {}

for sym in SYMBOLS:
    print(f"\n=== {sym} ===")
    if not mt5.symbol_select(sym, True):
        print(f"  skip: cannot select {sym}"); continue
    info = mt5.symbol_info(sym)
    if info is None:
        print(f"  skip: no symbol_info for {sym}"); continue
    point_size, spread_pts = info.point, info.spread

    df = get_mt5_data(sym, TIMEFRAME, N_CANDLES)
    if df is None or len(df) < 3000:
        print("  skip: insufficient data"); continue

    train_df, test_df = split_data(df, train_pct=TRAIN_PCT)

    fit = fit_strategy(train_df, point_size, spread_pts, compute_strategy_signals, suggest_params,
                       map_params, VARIANTS, STOP_MODES, n_trials=N_TRIALS_PER_VARIANT, seed=SEED,
                       min_active_days=MIN_ACTIVE_DAYS, graded=GRADED_PENALTY)
    if fit is None or fit["fitness"] <= MIN_IS_FITNESS:
        print(f"  no viable in-sample strategy, skipping OOS"); continue
    best, stop, ref_atr = fit["params"], fit["stop"], fit["ref_atr"]
    print(f"  WINNER {best['variant']} | stop {stop} | IS fitness {fit['fitness']:.4f}")

    # Blind OOS test, run ONCE; compute signals on full history so pre-market windows are warm.
    pos_oos = compute_strategy_signals(df, **map_params(best)).loc[test_df.index]
    oos_net = run_backtest(test_df, pos_oos, point_size, spread_pts, ref_atr, stop)
    oos = calculate_ai_metrics(oos_net, mc_simulations=500)
    if oos is None:
        print("  OOS metrics unavailable"); continue

    # Cost stress on the OOS slice: same positions + stop, harsher costs.
    swap = get_swap_pct(info, float(train_df["close"].median()), default=SWAP_DEFAULT)
    stress_tbl, stress = stress_test(test_df, pos_oos, point_size, spread_pts, ref_atr, stop,
                                     swap_pct_per_night=swap)
    stress_tables[sym] = stress_tbl

    bh = calculate_ai_metrics(test_df["close"].pct_change().dropna(), mc_simulations=100)
    generate_tearsheet(oos_net, title=f"{sym} {best['variant']} {stop['mode']} OOS")

    portfolio_results.append({
        "symbol": sym, "variant": best["variant"],
        "stop_mode": stop["mode"], "stop_atr_mult": stop["mult"],
        "is_fitness": fit["fitness"],
        "oos_cagr": oos["cagr"], "oos_sharpe": oos["sharpe"],
        "oos_sortino": oos["sortino"], "oos_max_dd": oos["max_drawdown"],
        "oos_calmar": oos["calmar"], "oos_win_rate": oos["win_rate"],
        "oos_profit_factor": oos["profit_factor"], "oos_active_days": oos["n_trades"],
        "oos_mc_worst5_dd": oos["mc_worst_5%_drawdown"],
        **stress, "swap_pct_night": swap,
        "bh_sharpe": bh["sharpe"] if bh else np.nan,
        "params": best,
    })

if portfolio_results:
    leaderboard = (pd.DataFrame(portfolio_results)
                     .sort_values("oos_sharpe", ascending=False).reset_index(drop=True))
    display(leaderboard)
else:
    print("No symbol produced a viable in-sample strategy.")

✅ Found 35 instruments in MT5 Market Watch.

=== BTCUSD ===
Data Split: 35000 Training Candles | 15000 Testing Candles
  IS fitness by group: {'mambafx_ny_breakout': 0.218}
  WINNER mambafx_ny_breakout | stop {'mode': 'fixed', 'mult': 3.564218501227791} | IS fitness 0.2180

=== USOIL ===
Data Split: 35000 Training Candles | 15000 Testing Candles
  IS fitness by group: {'mambafx_ny_breakout': -0.1915}
  no viable in-sample strategy, skipping OOS

=== XNGUSD ===
Data Split: 35000 Training Candles | 15000 Testing Candles
  IS fitness by group: {'mambafx_ny_breakout': 0.0407}
  WINNER mambafx_ny_breakout | stop {'mode': 'fixed', 'mult': 2.2451756438963817} | IS fitness 0.0407

=== AUDCAD ===
Data Split: 35000 Training Candles | 15000 Testing Candles
  IS fitness by group: {'mambafx_ny_breakout': -0.0039}
  no viable in-sample strategy, skipping OOS

=== AUDJPY ===
Data Split: 35000 Training Candles | 15000 Testing Candles
  IS fitness by group: {'mambafx_ny_breakout': -0.0357}
  no viable 

,symbol,variant,stop_mode,stop_atr_mult,is_fitness,oos_cagr,oos_sharpe,oos_sortino,oos_max_dd,oos_calmar,oos_win_rate,oos_profit_factor,oos_active_days,oos_mc_worst5_dd,stress_worst_cagr,stress_cagr_harsh,survives_stress,swap_pct_night,bh_sharpe,params
0,XNGUSD,mambafx_ny_breakout,fixed,2.245176,0.040743,0.220350,1.713967,6.309855,-0.022775,9.674897,0.204082,1.589260,49,-0.067680,0.188379,0.188379,True,0.009949,1.195858,"{'variant': 'mambafx_ny_breakout', 'entry_buff..."
1,JP225,mambafx_ny_breakout,fixed,1.055382,0.100408,0.068816,1.287552,4.614356,-0.018001,3.822894,0.187500,1.410511,48,-0.035199,0.068816,0.068816,True,0.000000,-0.504280,"{'variant': 'mambafx_ny_breakout', 'entry_buff..."
2,EURGBP,mambafx_ny_breakout,fixed,1.003208,0.007189,0.002641,0.463574,1.615000,-0.001445,1.827422,0.116279,1.142932,43,-0.004731,0.002641,0.002641,True,0.000045,-3.156611,"{'variant': 'mambafx_ny_breakout', 'entry_buff..."
3,BTCUSD,mambafx_ny_breakout,fixed,3.564219,0.217978,-0.009297,-0.047116,-0.141505,-0.045402,-0.204773,0.163265,0.987953,49,-0.067586,-0.009297,-0.009297,False,0.000277,0.242906,"{'variant': 'mambafx_ny_breakout', 'entry_buff..."
4,HK50,mambafx_ny_breakout,fixed,1.005225,0.090853,-0.013805,-0.656113,-1.323793,-0.017389,-0.793892,0.258621,0.881677,58,-0.027102,-0.014850,-0.014850,False,0.000090,-2.489840,"{'variant': 'mambafx_ny_breakout', 'entry_buff..."
5,UK100,mambafx_ny_breakout,fixed,1.004752,0.050745,-0.030442,-2.507067,-4.340262,-0.010345,-2.942694,0.195652,0.600698,46,-0.021625,-0.031476,-0.031476,False,0.000165,-0.610003,"{'variant': 'mambafx_ny_breakout', 'entry_buff..."
6,GBPUSD,mambafx_ny_breakout,fixed,4.417112,0.014252,-0.033762,-3.457681,-4.881488,-0.010170,-3.319864,0.184211,0.468840,38,-0.018806,-0.033762,-0.033762,False,0.000015,-1.360270,"{'variant': 'mambafx_ny_breakout', 'entry_buff..."


In [9]:
# ------------- WALK-FORWARD on the shortlist (re-optimises every fold: slow, run on candidates only) -------------
from functools import partial

WF_FOLDS       = 5
WF_INIT_TRAIN  = 0.5        # first 50% = first training set; the rest is split into WF_FOLDS test chunks
WF_ANCHORED    = True       # True = expanding train window, False = rolling window of the initial size
WF_TRIALS      = 30         # trials per variant per fold (5 folds x 4 variants x 30 = 600 backtests / symbol)
WF_MIN_POS_FOLDS = 0.8     # share of folds that must be profitable for wf_pass (0.8 = 4 of 5; stricter = fewer false positives)

# Candidates that were profitable OOS AND survived every cost scenario. Or set a list by hand.
SHORTLIST = (leaderboard.loc[leaderboard["survives_stress"] & (leaderboard["oos_sharpe"] > 0), "symbol"]
                        .head(5).tolist() if portfolio_results else [])
# SHORTLIST = ["XNGUSD", "EURJPY"]

wf_rows, wf_results = [], {}
for sym in SHORTLIST:
    print(f"\n=== WALK-FORWARD {sym} ===")
    mt5.symbol_select(sym, True)
    info = mt5.symbol_info(sym)
    point_size, spread_pts = info.point, info.spread
    df = get_mt5_data(sym, TIMEFRAME, N_CANDLES)

    swap = get_swap_pct(info, float(df["close"].median()), default=SWAP_DEFAULT)
    fit_fn = partial(fit_strategy, point_size=point_size, spread_pts=spread_pts,
                     signal_fn=compute_strategy_signals, space_fn=suggest_params, map_fn=map_params,
                     groups=VARIANTS, stop_modes=STOP_MODES, n_trials=WF_TRIALS, seed=SEED,
                     min_active_days=MIN_ACTIVE_DAYS, graded=GRADED_PENALTY, verbose=False)
    wf = walk_forward(df, fit_fn, compute_strategy_signals, map_params, point_size, spread_pts,
                      n_folds=WF_FOLDS, init_train_frac=WF_INIT_TRAIN, anchored=WF_ANCHORED,
                      min_is_fitness=MIN_IS_FITNESS, min_pos_fold_share=WF_MIN_POS_FOLDS,
                      engine_kw={"swap_pct_per_night": swap})   # walk-forward already includes swap
    wf_results[sym] = wf
    display(wf["folds"].drop(columns="params"))
    wf_rows.append({"symbol": sym, **wf["summary"]})
    generate_tearsheet(wf["returns"], title=f"{sym} walk-forward stitched OOS")

if wf_rows:
    wf_summary = pd.DataFrame(wf_rows).sort_values("wf_sharpe", ascending=False).reset_index(drop=True)
    display(wf_summary)
else:
    print("Empty shortlist - nothing survived the OOS + cost-stress screen.")



=== WALK-FORWARD XNGUSD ===


,fold,train_bars,test_start,test_end,viable,variant,stop_mode,stop_mult,is_fitness,oos_return,oos_sharpe,oos_max_dd,oos_active_days
0,1,25000,2026-06-18 18:05:00,2026-07-15 15:30:00,True,mambafx_ny_breakout,fixed,1.366783,0.188219,-0.027080,-15.983110,-0.027080,15
1,2,30000,2026-07-15 15:35:00,2026-08-11 03:05:00,True,mambafx_ny_breakout,fixed,2.620863,0.043151,-0.013905,-2.699132,-0.020460,15
2,3,35000,2026-08-11 03:10:00,2026-09-04 13:15:00,False,None,None,NaN,-0.047216,0.000000,NaN,NaN,0
3,4,40000,2026-09-04 13:20:00,2026-10-01 04:40:00,False,None,None,NaN,-0.009168,0.000000,NaN,NaN,0
4,5,45000,2026-10-01 04:45:00,2026-10-27 15:20:00,True,mambafx_ny_breakout,fixed,2.991878,0.131243,0.026979,2.267506,-0.015259,15



=== WALK-FORWARD JP225 ===


,fold,train_bars,test_start,test_end,viable,variant,stop_mode,stop_mult,is_fitness,oos_return,oos_sharpe,oos_max_dd,oos_active_days
0,1,25000,2026-06-22 13:55:00,2026-07-16 20:40:00,True,mambafx_ny_breakout,fixed,1.161597,0.132913,0.015248,2.680991,-0.010746,17
1,2,30000,2026-07-16 20:45:00,2026-08-12 01:10:00,True,mambafx_ny_breakout,fixed,1.050674,0.125903,0.000161,0.056875,-0.009365,17
2,3,35000,2026-08-12 01:15:00,2026-09-07 04:10:00,True,mambafx_ny_breakout,fixed,1.048273,0.073897,0.013217,2.407654,-0.011962,17
3,4,40000,2026-09-07 04:15:00,2026-10-01 11:05:00,True,mambafx_ny_breakout,fixed,1.067100,0.076118,-0.004194,-1.482781,-0.009900,16
4,5,45000,2026-10-01 11:10:00,2026-10-27 15:20:00,True,mambafx_ny_breakout,fixed,1.067100,0.044180,0.024461,2.827376,-0.009403,17



=== WALK-FORWARD EURGBP ===


,fold,train_bars,test_start,test_end,viable,variant,stop_mode,stop_mult,is_fitness,oos_return,oos_sharpe,oos_max_dd,oos_active_days
0,1,25000,2026-06-26 16:50:00,2026-07-22 01:45:00,False,None,None,NaN,-0.010999,0.000000,NaN,NaN,0
1,2,30000,2026-07-22 01:50:00,2026-08-14 10:45:00,True,mambafx_ny_breakout,fixed,1.005971,0.000220,0.002810,3.751140,-0.000901,16
2,3,35000,2026-08-14 10:50:00,2026-09-08 19:50:00,True,mambafx_ny_breakout,fixed,1.012221,0.006711,0.001181,1.552869,-0.001458,15
3,4,40000,2026-09-08 19:55:00,2026-10-02 05:10:00,True,mambafx_ny_breakout,fixed,1.073277,0.006753,0.001321,2.267713,-0.000746,14
4,5,45000,2026-10-02 05:15:00,2026-10-27 15:20:00,True,mambafx_ny_breakout,fixed,1.059880,0.002954,-0.000875,-2.311606,-0.001328,14


,symbol,wf_total_return,wf_cagr,wf_sharpe,wf_max_dd,folds_viable,folds_positive,pct_folds_positive,median_fold_return,worst_fold_return,variant_stability,wf_pass
0,EURGBP,0.004441,0.011460,1.765827,-0.001458,4,3,0.6,0.001181,-0.000875,1.0,False
1,JP225,0.049578,0.100775,1.759694,-0.014424,5,4,0.8,0.013217,-0.004194,1.0,True
2,XNGUSD,-0.014726,-0.028134,-0.472451,-0.047219,3,1,0.2,0.000000,-0.027080,1.0,False


In [ ]:
mt5.shutdown()
print("MT5 connection closed.")
